# `treino_rede` — Treinamento da Rede Hopfield (máquina de treinamento)

Notebook focado **exclusivamente no treinamento**: assume que os arquivos de
pré-processamento já existem (binarização, alinhamento, seleção de genes,
projeção SWeeP) e executa apenas as etapas de extração de padrões e
armazenamento na rede.

**Saídas geradas:**
- `outputs/hopfield/rede35_v2.pt` — rede treinada (padrões + hiperparâmetros)
- `outputs/hopfield/rede35_v2_metadata.json` — metadados para avaliação

**Copiar esses dois arquivos para a máquina de aplicação** e executar as
seções 10b em diante do `pipeline_hopfield_v2.ipynb`.

## 1. Imports e verificação das entradas

In [ ]:
import sys, os, json, importlib
import numpy as np
import torch

SRC_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import config; importlib.reload(config)
from config import (
    PATH_LABELS_F, PATH_SWEEP_F,
    OUT_TOP_GENES, OUT_TREINAMENTO, OUT_HOPFIELD,
)
from treinamento import (
    CarregadorDadosFujita, ProjetorSWeP,
    ExtratorPadroesSubcluster, ModernHopfieldNetwork,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.cuda.manual_seed_all(SEED)

In [ ]:
path_f_top5k = os.path.join(OUT_TREINAMENTO, 'adataF_binarizado_alinhado_top5000.txt')
path_top5k   = os.path.join(OUT_TOP_GENES,   'top5000_frequentes.csv')

entradas = {
    'Matriz Fujita top5000': path_f_top5k,
    'SWeeP pré-computado'  : PATH_SWEEP_F,
    'Top 5000 genes'       : path_top5k,
    'Labels Fujita'        : PATH_LABELS_F,
}
tudo_ok = True
for nome, path in entradas.items():
    ok = os.path.exists(path)
    status = '✓' if ok else '✗ NÃO ENCONTRADO'
    print(f'  {nome:30s} : {status}')
    if not ok:
        tudo_ok = False

if not tudo_ok:
    raise FileNotFoundError('Arquivos de pré-processamento faltando. '
                             'Execute pipeline_hopfield_v2.ipynb (seções 1-5) primeiro.')

## 2. Carregamento dos dados Fujita

In [ ]:
carregador = CarregadorDadosFujita(
    path_matriz = path_f_top5k,
    path_genes  = path_top5k,
    path_labels = PATH_LABELS_F,
    path_sweep  = PATH_SWEEP_F,
    n_genes     = 5000,
).carregar()
print(carregador)

## 3. Remapeamento de classes (clo)

Classes não presentes em `[1, 3, 4, 5, 6, 7]` são remapeadas para `2`
(padrão do script01 MATLAB original).

In [ ]:
clo = carregador.labels.copy()
clo[~np.isin(clo, [1, 3, 4, 5, 6, 7, 0])] = 2

vals, counts = np.unique(clo, return_counts=True)
print('Distribuição de classes (clo):')
for v, c in zip(vals, counts):
    print(f'  classe {v}: {c:>6d} células')

## 4. PCA no espaço SWeeP

In [ ]:
projetor = ProjetorSWeP(n_features=5000, n_componentes=600, seed=SEED)
projetor.usar_sweep_precomputado(carregador.Wswp).aplicar_pca()
print(projetor)

## 5. Extração de padrões por subcluster (K-means)

7 classes × 10 clusters = **70 padrões**

In [ ]:
extrator = ExtratorPadroesSubcluster(
    W0      = carregador.W0,
    labels  = clo,
    classes = [1, 2, 3, 4, 5, 6, 7],
    nc      = 10,
    seed    = SEED,
    k       = 1,
).extrair(projetor.Wswp)

perf35 = extrator.padroes
print(extrator)
print(f'perf35 shape: {perf35.shape}  (esperado: (70, 5000))')

## 6. Treinamento da rede

In [ ]:
rede35 = ModernHopfieldNetwork(beta=8.0, n_iters=1, binary=True, threshold=0.5)
rede35.store(perf35)
print(rede35)

## 7. Exportação — rede + metadados

Gera dois arquivos para copiar à máquina de aplicação:
- **`rede35_v2.pt`** — rede treinada (padrões em `{-1,+1}` + hiperparâmetros)
- **`rede35_v2_metadata.json`** — classes, nc e mapeamento padrão → classe

Esses dois arquivos são suficientes para executar a recuperação e avaliação
sem nenhum dado de treinamento adicional.

In [ ]:
os.makedirs(OUT_HOPFIELD, exist_ok=True)
path_pt   = os.path.join(OUT_HOPFIELD, 'rede35_v2.pt')
path_meta = os.path.join(OUT_HOPFIELD, 'rede35_v2_metadata.json')

# Rede: hiperparâmetros + padrões armazenados em {-1,+1}
rede35.salvar(path_pt)

# Metadados: necessários para o AvaliadorHopfield na máquina de aplicação
metadata = {
    'classes'   : [1, 2, 3, 4, 5, 6, 7],
    'nc'        : 10,
    'n_patterns': int(perf35.shape[0]),
    'n_genes'   : int(perf35.shape[1]),
    'meta'      : [[int(c), int(idx)] for c, idx in extrator.meta],
}
with open(path_meta, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'\nArquivos exportados para copiar à máquina de aplicação:')
for p in [path_pt, path_meta]:
    size = os.path.getsize(p) / 1e6
    print(f'  {os.path.basename(p):35s}  {size:.1f} MB')
print(f'\nTotal: {sum(os.path.getsize(p) for p in [path_pt, path_meta]) / 1e6:.1f} MB')